# 🎙️ speakerscribe — Speech-to-Text with Speaker Diarization

**Transcribe audio/video files with automatic speaker identification.**

Powered by [faster-whisper](https://github.com/SYSTRAN/faster-whisper) + [pyannote.audio](https://github.com/pyannote/pyannote-audio).

---

## ✅ Before you start — checklist

1. **GPU runtime**: `Runtime → Change runtime type → T4 GPU`
2. **HuggingFace token** (required for speaker diarization):
   - Create a **Read** token at https://huggingface.co/settings/tokens
   - Accept the model terms at https://huggingface.co/pyannote/speaker-diarization-community-1
   - Add it to **Colab Secrets** (🔑 icon in the left sidebar) with the name `HF_TOKEN`
3. **Place your audio/video files** in your Google Drive, inside a folder called `data/` within your project workspace.

---

> 💡 **Supported formats:** `.mp4` `.mp3` `.wav` `.m4a` `.mkv` `.aac` `.flac` `.ogg` `.webm`

## Step 1 — Install speakerscribe

> ⚠️ After the cell finishes, **restart the runtime** (`Runtime → Restart runtime`) and then run all remaining cells.

In [ ]:
# # Install speakerscribe and its dependencies
# # This takes 3-5 minutes on the first run (downloading model weights separately)
# %pip install speakerscribe --quiet

# print("✅ Installation complete.")
# print("⚠️  Please restart the runtime now: Runtime → Restart runtime")
# print("   Then run all remaining cells (do NOT re-run this cell).")

## Step 2 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted at /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/Pruebas/Speakerscribe"

/content/drive/MyDrive/Pruebas/Speakerscribe


In [3]:
%ls

 CHANGELOG.md   pyproject.toml   speakerscribe_colab.ipynb
 data/          README.md        tests/
 LICENSE        speakerscribe/  'z. Backups'/


## Importar/Instalar paquete

In [4]:
# ── Instalar speakerscribe desde Google Drive (sin subir a PyPI) ──────────────
# Modo editable: cualquier cambio en los .py se refleja sin reinstalar.
# Corre esta celda UNA sola vez. Luego reinicia el runtime si es la primera vez.

import subprocess, sys

DRIVE_PKG = "/content/drive/MyDrive/Pruebas/Speakerscribe"

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", DRIVE_PKG, "--quiet"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ speakerscribe instalado desde Drive (modo editable)")
    print(f"   Ruta: {DRIVE_PKG}")
else:
    print("❌ Error al instalar:")
    print(result.stderr)

# Verificar que importa bien
import importlib
import speakerscribe
importlib.reload(speakerscribe)
print(f"   Versión: {speakerscribe.__version__}")
print("✅ Installation complete.")
print("⚠️  Please restart the runtime now: Runtime → Restart runtime")
print("   Then run all remaining cells (do NOT re-run this cell).")

✅ speakerscribe instalado desde Drive (modo editable)
   Ruta: /content/drive/MyDrive/Pruebas/Speakerscribe
   Versión: 0.1.0


## Step 3 — Configuration

**Only edit the cells in this section.**

In [4]:
# ══════════════════════════════════════════════════════════════════
#  ✏️  EDIT HERE — all other cells can run without changes
# ══════════════════════════════════════════════════════════════════

# Path to your project workspace in Google Drive.
# Place your audio/video files inside <WORKSPACE>/data/
WORKSPACE = "/content/drive/MyDrive/Pruebas/Speakerscribe"

# Whisper model. Options (GPU VRAM required):
#   'tiny'           (~1.5 GB)  — fastest, basic quality
#   'small'          (~2.5 GB)  — fast, decent quality
#   'large-v3-turbo' (~3.5 GB)  — recommended: fast + excellent quality ⭐
#   'large-v3'       (~5.0 GB)  — best quality, slower
MODEL = "large-v3-turbo"

# Language code (None = auto-detect). Examples: 'en', 'es', 'fr', 'pt'
LANGUAGE = None

# Speaker diarization (requires HF_TOKEN in Colab Secrets)
ENABLE_DIARIZATION = True

# Optional: pin the exact number of speakers if you know it.
# Leave as None if you don't know in advance.
NUM_SPEAKERS = None   # e.g. 2 for a one-on-one interview

# Glossary / initial prompt: helps with proper nouns, brand names, jargon.
# Leave empty string for no prompt.
INITIAL_PROMPT = ""

VAD_MIN_SILENCE_MS = 1000       # ms — sube para segmentos más largos (default: 500)
GAP_MAX_S          = 3.0        # seg — pausa máxima antes de abrir nuevo bloque en .md

# Force re-processing of files even if outputs already exist.
FORCE_REPROCESS = False

# ══════════════════════════════════════════════════════════════════
print(f"Workspace : {WORKSPACE}")
print(f"Model     : {MODEL}")
print(f"Language  : {LANGUAGE or 'auto-detect'}")
print(f"Diarize   : {ENABLE_DIARIZATION}")
print(f"Speakers  : {NUM_SPEAKERS or 'auto-detect'}")

Workspace : /content/drive/MyDrive/Pruebas/Speakerscribe
Model     : large-v3-turbo
Language  : auto-detect
Diarize   : True
Speakers  : auto-detect


## Step 4 — Pre-flight check

Validates GPU, disk space, HF token, and detects your audio files — before loading any models.

In [5]:
import speakerscribe
from speakerscribe import TranscriptionConfig, WorkspacePaths
from speakerscribe.pipeline import preflight_check

print(f"speakerscribe version: {speakerscribe.__version__}")

config = TranscriptionConfig(
    model=MODEL,
    language=LANGUAGE,
    beam_size=5,
    enable_diarization=ENABLE_DIARIZATION,
    num_speakers=NUM_SPEAKERS,
    min_speakers=MIN_SPEAKERS,
    max_speakers=MAX_SPEAKERS,
    initial_prompt=INITIAL_PROMPT or None,
    vad_min_silence_ms=VAD_MIN_SILENCE_MS,
    gap_max_s_transcript=GAP_MAX_S,
    force_reprocess=FORCE_REPROCESS,
    evaluate_quality=True,
)

paths = WorkspacePaths(workspace=WORKSPACE)
paths.create_directories()

# Show what files are in data/
media = paths.list_media_files()
print(f"\n📂 Files found in {paths.data}:")
for i, f in enumerate(media, 1):
    size_mb = f.stat().st_size / 1e6
    print(f"   {i}. {f.name}  ({size_mb:.1f} MB)")

if not media:
    print("\n⚠️  No audio files found!")
    print(f"   Place your files inside: {paths.data}")
else:
    # Run pre-flight checks
    info = preflight_check(paths, config)
    print(f"\n✅ Pre-flight OK — {info['n_files']} file(s) ready to process")

2026-05-01 13:44:05.151 | INFO     | speakerscribe.pipeline:preflight_check:58 - Pre-flight check...
2026-05-01 13:44:05.156 | INFO     | speakerscribe.pipeline:preflight_check:69 -    1 file(s) — 60.0 MB total
2026-05-01 13:44:05.158 | INFO     | speakerscribe.pipeline:preflight_check:79 -    Disk: 70,301 MB free (~500 MB required)


speakerscribe version: 0.1.0

📂 Files found in /content/drive/MyDrive/Pruebas/Speakerscribe/data:
   1. 2026-04-30 *Entrevista Cooperación Suiza.mp4  (60.0 MB)


2026-05-01 13:44:10.514 | INFO     | speakerscribe.pipeline:preflight_check:106 -    VRAM: 15.6 GB free / 15.6 GB total
2026-05-01 13:44:10.515 | INFO     | speakerscribe.pipeline:preflight_check:112 -    Device: cuda (float16)
2026-05-01 13:44:10.947 | INFO     | speakerscribe.pipeline:preflight_check:129 -    HF token: detected (hf_Ioypb...)



✅ Pre-flight OK — 1 file(s) ready to process


## Step 5 — Transcribe

Processes all files in `<WORKSPACE>/data/`. The Whisper model is loaded once and reused for all files.

In [6]:
%%time
from speakerscribe import process_batch

results = process_batch(paths, config)

# Summary
n_ok    = sum(1 for r in results if r.get("status") == "ok")
n_skip  = sum(1 for r in results if r.get("status") == "skipped")
n_err   = sum(1 for r in results if r.get("status") == "error")
n_words = sum(r.get("total_words", 0) for r in results if r.get("status") == "ok")

print(f"\n🎉 Done!  OK={n_ok}  Skipped={n_skip}  Errors={n_err}  Total words={n_words:,}")
print(f"\n📁 Outputs saved to: {paths.transcripts}")
print(f"📁 Splits saved to  : {paths.splits}")

2026-05-01 13:44:39.816 | INFO     | speakerscribe.pipeline:preflight_check:58 - Pre-flight check...
2026-05-01 13:44:39.822 | INFO     | speakerscribe.pipeline:preflight_check:69 -    1 file(s) — 60.0 MB total
2026-05-01 13:44:39.823 | INFO     | speakerscribe.pipeline:preflight_check:79 -    Disk: 70,300 MB free (~500 MB required)
2026-05-01 13:44:39.825 | INFO     | speakerscribe.pipeline:preflight_check:106 -    VRAM: 15.6 GB free / 15.6 GB total
2026-05-01 13:44:39.826 | INFO     | speakerscribe.pipeline:preflight_check:112 -    Device: cuda (float16)
2026-05-01 13:44:40.205 | INFO     | speakerscribe.pipeline:preflight_check:129 -    HF token: detected (hf_Ioypb...)
2026-05-01 13:44:40.207 | INFO     | speakerscribe.pipeline:process_batch:343 - 1 file(s) detected:
2026-05-01 13:44:40.209 | INFO     | speakerscribe.pipeline:process_batch:345 -    1. 2026-04-30 *Entrevista Cooperación Suiza.mp4 (60.0 MB)
2026-05-01 13:44:55.534 | INFO     | speakerscribe.transcription:load_whisper

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

embedding/pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

2026-05-01 13:45:27.860 | DEBUG    | speakerscribe.diarization:diarize_audio:84 - VRAM after loading pyannote: 0.03 GB
2026-05-01 13:45:27.861 | INFO     | speakerscribe.diarization:diarize_audio:101 -    Speakers: auto-detect
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = 


🎉 Done!  OK=1  Skipped=0  Errors=0  Total words=4,073

📁 Outputs saved to: /content/drive/MyDrive/Pruebas/Speakerscribe/transcripts
📁 Splits saved to  : /content/drive/MyDrive/Pruebas/Speakerscribe/splits
CPU times: user 5min 49s, sys: 57.6 s, total: 6min 46s
Wall time: 7min 29s


## Step 6 (optional) — Inspect a transcription

In [ ]:
from speakerscribe import inspect_json
from pathlib import Path

# Change this to the actual JSON file you want to inspect
JSON_FILE = str(paths.transcripts) + "/YOUR_FILE_large-v3-turbo.json"

if Path(JSON_FILE).exists():
    info = inspect_json(Path(JSON_FILE))
else:
    # List available files
    json_files = list(paths.transcripts.glob("*.json"))
    if json_files:
        print("Available JSON files:")
        for f in json_files:
            print(f"  {f.name}")
        # Auto-inspect the first one
        print("\n--- Inspecting first file ---")
        info = inspect_json(json_files[0])
    else:
        print("No JSON files found yet. Run Step 5 first.")

## Step 7 (optional) — Rename speaker labels

After reading the transcript and identifying who is who, replace `SPEAKER_00`, `SPEAKER_01`, etc. with real names.

In [ ]:
from speakerscribe import rename_speakers_in_outputs

# ✏️ Edit these values:
BASE_NAME = "your_audio_file_large-v3-turbo"  # filename stem + model name (no extension)
SPEAKER_MAPPING = {
    "SPEAKER_00": "Alice",
    "SPEAKER_01": "Bob",
    # Add more speakers as needed
}

stats = rename_speakers_in_outputs(paths, BASE_NAME, SPEAKER_MAPPING)
print(f"\n✅ Renamed speakers in {len(stats)} file(s)")
for filename, count in stats.items():
    print(f"   {filename}: {count} replacements")

## Step 8 (optional) — Clean outputs

Remove outputs for a specific file so you can re-process it with different settings.

In [ ]:
from speakerscribe import delete_outputs_for

# ✏️ Set a pattern that matches the filename(s) you want to remove
PATTERN = "your_audio_file"  # deletes all outputs that contain this substring

n_deleted = delete_outputs_for(paths, pattern=PATTERN, include_diar_cache=True)
print(f"✅ {n_deleted} file(s) deleted — ready to re-process")

## 🛑 8. Apagar la sesión (libera cuota Colab)

**IMPORTANTE:** correr esta celda al final ahorra cuota de GPU. Colab cobra cuota mientras la sesión está activa, incluso si no estás procesando nada.

**Opciones:**
1. **Liberar memoria sin cerrar:** útil si planeás seguir trabajando después.
2. **Apagar runtime completo:** termina la sesión, libera la VRAM/RAM y deja de consumir cuota.

> ⚠️ Tras apagar el runtime, vas a tener que re-correr la sección 1 y 2 antes de procesar nuevamente.


In [ ]:
# ── 8.1 Liberar memoria SIN cerrar la sesión ──────────────────────
# Útil si planeás seguir trabajando. Si vas a cerrar el notebook,
# usá la celda 8.2 abajo (apaga el runtime y libera cuota).

# Liberar variables del run
for var_nombre in ["modelo", "modelo_test", "resultados", "resultado"]:
    if var_nombre in globals():
        del globals()[var_nombre]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("🧹 Memoria liberada. La sesión sigue activa.")
print("   Si terminaste de trabajar, corré la celda 8.2 para apagar el runtime.")


In [ ]:
# ── 8.2 Apagar el runtime de Colab (libera cuota de GPU) ──────────
# CONFIRMACIÓN: descomentá la última línea SOLO cuando estés seguro
# de que terminaste. Esta celda termina la sesión y libera la GPU.
#
# Tras apagar, vas a tener que re-correr la sección 1 y 2 si volvés.

print("⚠️  ESTA CELDA TERMINA LA SESIÓN. Descomentá la última línea para ejecutar.")
print(f"   Hora actual Bogotá: {datetime.now(pytz.timezone('America/Bogota')).strftime('%Y-%m-%d %H:%M:%S')}")

# Liberar TODO antes de apagar
for var_nombre in list(globals().keys()):
    if var_nombre.startswith("modelo") or var_nombre.startswith("resultado"):
        try:
            del globals()[var_nombre]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── DESCOMENTÁ LA LÍNEA SIGUIENTE PARA APAGAR ─────────────────────
# from google.colab import runtime; runtime.unassign()

print()
print("🛑 Para apagar AHORA, descomentá la línea: runtime.unassign()")
print("   Después de apagar, recargá la página para volver a usar Colab.")


---

## 📖 Tips

- **Colab session reset?** Re-run Steps 2–5. Outputs on Drive are safe.
- **Already processed?** speakerscribe skips files with existing outputs unless `FORCE_REPROCESS = True`.
- **Diarization cache** is saved in `_diar_cache/`. If you re-run with the same audio, pyannote is not called again.
- **Splits** in `splits/` are ready to paste into ChatGPT, Claude, or any other LLM for summarization.
- **Quality warnings** in the log? Check the `REPETITIONS` flag — if present, try reducing `beam_size` or adding an `initial_prompt`.

---

📦 [speakerscribe on PyPI](https://pypi.org/project/speakerscribe/) ·
📁 [GitHub](https://github.com/EnriqueForero/speakerscribe) ·
🐛 [Report an issue](https://github.com/EnriqueForero/speakerscribe/issues)